# 08 — Structured Pruning

Structured pruning removes entire heads, layers, or neurons, yielding real speedups (unlike unstructured weight pruning).

## Head Pruning, Layer Dropping, and Width Pruning

**Unstructured pruning** (set individual weights to zero) achieves high compression ratios but requires sparse matrix support for speedups. Most hardware accelerators prefer dense operations.

**Structured pruning** removes entire computational units:
- *Head pruning*: remove attention heads with low importance scores
- *Layer dropping*: remove entire transformer layers
- *Width pruning*: remove neurons from FFN or reduce embedding dimension

**Importance scoring for heads** (Michel et al., 2019):
- Gradient-based: importance ≈ *|W_O ∂L/∂W_O|* (expected sensitivity)
- Taylor expansion: approximate the change in loss from zeroing each head
- Attention entropy: heads with uniform attention (high entropy) contribute less

**The lottery ticket hypothesis** (Frankle & Carlin, 2019) suggests sparse subnetworks ('winning tickets') exist from initialization that can be trained to full performance. But finding them requires iterative magnitude pruning + retraining — expensive.

**Practical recipe**: structured pruning → fine-tune → structured pruning → fine-tune (iterative). Each pruning step removes the least important structures; fine-tuning recovers lost performance.

In [ ]:
# Attention head importance + pruning
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification

torch.manual_seed(42)

class MultiHeadAttnPrunable(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.d_model = d_model
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        # Pruning mask (1=active, 0=pruned)
        self.register_buffer('head_mask', torch.ones(n_heads))

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2,-1) / self.d_head**0.5).softmax(dim=-1)
        out = (attn @ v) * self.head_mask.reshape(1, -1, 1, 1)
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.out(out)

    def head_importance_scores(self, x, labels, loss_fn):
        """Compute gradient-based head importance."""
        self.head_mask.requires_grad_(True)
        out = self.forward(x)
        # Fake 2-layer net for demo
        logits = torch.nn.functional.linear(out.mean(1),
                                              torch.randn(5, self.d_model))
        loss = loss_fn(logits, labels)
        loss.backward()
        importance = self.head_mask.grad.abs()
        self.head_mask.requires_grad_(False)
        return importance.detach()


attn = MultiHeadAttnPrunable(d_model=32, n_heads=4)

# Generate dummy data and compute head importance
x = torch.randn(8, 16, 32)
labels = torch.randint(0, 5, (8,))

importance = attn.head_importance_scores(x.clone(), labels, nn.CrossEntropyLoss())
print('Head importance scores:')
for i, imp in enumerate(importance):
    print(f'  Head {i}: {imp.item():.4f}')

# Prune the least important head
sorted_heads = importance.argsort()
attn.head_mask[sorted_heads[0]] = 0.0  # prune lowest importance head
print(f'\nPruned head {sorted_heads[0].item()} (importance: {importance[sorted_heads[0]].item():.4f})')

y_pruned = attn(x)
print(f'Output after pruning: {y_pruned.shape}')